In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier  
from sklearn.preprocessing import LabelEncoder
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# the path where the project root is at
BASE_ENV = Path().resolve().parent

In [ ]:
df = pd.read_csv(BASE_ENV / 'data/raw/CICFlowMeter_out.csv')
df.head()

In [ ]:
columns_to_drop = []

In [ ]:
# current dataset's column name
df.columns.tolist()

In [ ]:
df.isnull().sum()

In [ ]:
nan_counts = np.isnan(df.select_dtypes(include=[np.number])).sum()

print("Columns containing nan:")
print(nan_counts[nan_counts > 0])

In [ ]:
inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()

print("Columns containing nan:")
print(inf_counts[inf_counts > 0])

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df = df.replace('-', np.nan)

In [ ]:
nan_counts = np.isnan(df.select_dtypes(include=[np.number])).sum()

print("Columns containing nan:")
print(nan_counts[nan_counts > 0])

In [ ]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('/', '_')
df.columns

In [ ]:
df['Label'].value_counts()

In [ ]:
# checking duplicates
duplicate_names = df.columns[df.columns.str.contains(r'\.\d+$')]
print("Duplicate column names:")
print(duplicate_names.tolist())

columns_to_drop.extend(duplicate_names.tolist())

In [ ]:
# checking hidden feature colnes 
hidden_clones = set()
for i in range(df.shape[1]):
    col_1 = df.iloc[:, i]
    
    for j in range(i + 1, df.shape[1]):
        col_2 = df.iloc[:, j]
        
        if col_1.equals(col_2):
            hidden_clones.add(df.columns[j])

print(f"clone columns: {hidden_clones}")
columns_to_drop.extend(list(hidden_clones))

In [ ]:
# checking logical negative values
negative_values = (df.select_dtypes(include=[np.number]) < 0).sum()
print("Columns containing negative values:")
print(negative_values[negative_values > 0])

In [ ]:
df.columns

In [ ]:

# 2. Split the dataset into two temporary DataFrames
df_normal = df[df['Label'] == 'Benign']
df_attacks = df[df['Label'] != 'Benign']

# Calculate total number of attacks (should be around 22,215 based on your image)
total_attacks = len(df_attacks)

print(f"Original Normal count: {len(df_normal)}")
print(f"Original Attack count: {total_attacks}")

# 3. Undersample the Normal traffic to a 2:1 ratio
# (If you want a 1:1 ratio, just remove the * 2)
df_normal_undersampled = df_normal.sample(n=total_attacks * 2, random_state=42)

# 4. Recombine the datasets into your new, balanced master DataFrame
df_balanced = pd.concat([df_normal_undersampled, df_attacks])

# 5. Shuffle the rows thoroughly so the model doesn't read all normal packets first

In [ ]:
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# drop noise
columns_to_drop.extend(['Flow_ID', 'Src_IP', 'Src_Port', 'Dst_IP', 'Dst_Port'])
df = df.drop(columns=columns_to_drop)

In [ ]:
df.head()

In [ ]:
# for testing
df = df.drop(columns='Timestamp')

## Correlation Testing

In [ ]:
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

In [ ]:
df['Label'].value_counts()

In [ ]:
X = df.copy()
y = X['Label']
X = X.drop(columns=['Label'])

In [ ]:
rf_checker = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_checker.fit(X, y)

In [ ]:
important_features = pd.Series(rf_checker.feature_importances_, index=X.columns)
important_features = important_features.sort_values(ascending=False)

In [ ]:
dead_correlation_columns_RF = important_features[important_features <= 0.001].to_dict()
print("Features with low importance according to Random Forest:")
print(dead_correlation_columns_RF)

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.barplot(x=list(dead_correlation_columns_RF.values()), y=list(dead_correlation_columns_RF.keys()), color='blue')
plt.title("Features with low importance according to Random Forest", fontsize=14, pad=15, fontweight='bold')
plt.xlabel("importance vs all features", fontsize=12)
plt.ylabel("Network Features", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
columns_to_drop.extend(list(dead_correlation_columns_RF.keys()))
columns_to_drop